<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# REINFORCE for _cartpole_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Based on:
> Udacity, Deep Reinforcement Learning Course. Available on [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/reinforce).

## Configuration

In [ ]:
%pip install "gymnasium[classic-control]"

In [2]:
# set render to 'rgb_array' (change to 'human' for local display-enabled runtime)
render_mode = "rgb_array"

## Environment

We will be using [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), a simple control problem.

<img src="https://github.com/jgromero/drl-csic/blob/main/code/cartpole.gif?raw=true"/>


The action and observation spaces can be printed as follows.

In [3]:
import gymnasium as gym
env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)


## REINFORCE algorithm
Next we provide a generic implementation of REINFORCE. To reduce the variance of the gradient estimate, each action is weighted by the (normalized) discounted return obtained from that step onwards, $G_t$, instead of by the total return of the episode. We do not use external files here. There is no explicit _agent_ object; its capabilities (i.e., act) are encoded along with the policy network.

The [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment is considered solved when the agent obtains an average score of at least $475$ points over 100 consecutive episodes. Episodes are truncated at $500$ steps, which is therefore the maximum score.

### Imports and initializations

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### Policy definition

In [5]:
class Policy(nn.Module):
    def __init__(self, s_size=4, h_size=16, a_size=2):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

### Training loop

In [ ]:
policy = Policy().to(device)

SOLVED_SCORE = 475.0   # average score (last 100 episodes) to consider CartPole-v1 solved

def reinforce(policy, n_episodes=1000, max_t=500, gamma=0.99, print_every=100):
    """REINFORCE.

    Params
    ======
        policy (Policy): neural network with act method
        n_episodes (int): maximum number of episodes (n_episodes)
        max_t (int): maximum number of steps per episode (n_steps)
        gamma (float): discount factor
        print_every (int): print score each print_every episodes
    """

    scores = []                          # scores for every episode
    scores_window = deque(maxlen=100)    # scores for the last 100 episodes

    optimizer = optim.Adam(policy.parameters(), lr=1e-2)  # optimizer used for training

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        saved_log_probs = []  # action probabilities
        rewards = []          # rewards of the trajectory

        for t in range(max_t):

            # choose action At with the policy network
            action, log_prob = policy.act(state)

            # save actions' probabilities
            saved_log_probs.append(log_prob)

            # apply At and get Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # store reward
            rewards.append(reward)

            # advance to next state
            state = next_state
            score += reward

            if terminated or truncated:
                break

        scores_window.append(score)       # save last score (last 100)
        scores.append(score)              # save last score (every episode)

        # calculate discounted return from each step onwards: G_t = R_t+1 + gamma * G_t+1
        returns = deque()
        G = 0
        for reward in reversed(rewards):
            G = reward + gamma * G
            returns.appendleft(G)

        # normalize returns (reduces the variance of the gradient estimate)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        # train & update
        policy_loss = []
        for log_prob, G in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * G)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        # print training info
        if i_episode % print_every == 0:
            print('Episode {}\tAverage score (last 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\nProblem solved in {:d} episodes!\tAverage score (last 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblem not solved in {:d} episodes.\tAverage score (last 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(policy.state_dict(), 'checkpoint-cartpole-reinforce.pth') # save weights for trained agent

    return scores

scores = reinforce(policy, n_episodes=5000)

# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

## Visualize trained agent

We can afterwards load the trained agent and visualize how it acts on the environment. (Local run is required for 2D rendering, not Google Colaboratory.)

In [ ]:
# load weights from `checkpoint-cartpole-reinforce.pth`
policy_load = Policy().to(device)

policy_load.load_state_dict(torch.load('checkpoint-cartpole-reinforce.pth', map_location=device, weights_only=True))

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action, _ = policy_load.act(state)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()